
# HOG + LBP + KAZE → SVM / Random Forest
## Göz Bölgesi (Combined Eye ROI) Deepfake Tespiti — Kader / Deney 1

Bu notebook, yüklenen **HOG + LBP + KAZE → SVM / Random Forest Mouth** deneyinin **Kader → Deney 1 → Göz** veri yapısına revize edilmiş sürümüdür.

### Deneyde karşılaştırılan üç model
1. **HOG + LBP → RBF-SVM**  
2. **HOG + LBP + KAZE → RBF-SVM**  
3. **HOG + LBP + KAZE → Random Forest**

### Temel kurallar
- Girdi yalnızca daha önce çıkarılmış **Combined Eye ROI** görüntüleridir.
- Kaynak ROI dosyaları değiştirilmez, silinmez veya yeniden adlandırılmaz.
- `real = 0`, `fake = 1`.
- Train / Validation / Test splitleri mevcut metadata'dan okunur; yeni rastgele split yapılmaz.
- StandardScaler yalnızca **train** üzerinde öğrenilir.
- Model ve karar threshold'u yalnızca **validation** üzerinde seçilir.
- Test seti hiperparametre/threshold seçimi için kullanılmaz.
- `video_id` alanı mevcut metadata'da orijinal kaynak video kimliğini kanıtlamıyorsa, gerçek video-level leakage sonucu **PASS** olarak uydurulmaz.
- Exact-content leakage için SHA256 kontrolü yapılır.
- Feature extraction sonuçları cache'e atomik olarak yazılır.
- Çıktılar yalnızca:
  `Kader/Deney 1/Sonuçlar/<run_id>/`
  altına kaydedilir.
- Run ID biçimi:
  `YYYYMMDD_HHMM_eye_hog_lbp_kaze_svm_rf_seed42`

> Bu notebook klasik makine öğrenmesi deneyidir. PyTorch optimizer/epoch/checkpoint state'i doğal olarak uygulanmaz. Bunun yerine scaler, model, threshold, config ve RNG state'i atomik olarak kaydedilip yeniden yükleme testi yapılır.


In [1]:
import cv2

print(cv2.__version__)
print(hasattr(cv2, "KAZE_create"))
print(hasattr(cv2, "AKAZE_create"))
print(cv2.getBuildInformation()[:500])

4.10.0
True
True

General configuration for OpenCV 4.10.0 =====================================
  Version control:               4.10.0-dirty

  Extra modules:
    Location (extra):            /io/opencv_contrib/modules
    Version control (extra):     4.10.0

  Platform:
    Timestamp:                   2024-06-17T17:56:43Z
    Host:                        Linux 5.15.0-1064-azure x86_64
    CMake:                       3.29.5
    CMake generator:             Unix Makefiles
    CMake build tool:            /bin/


In [2]:
!pip uninstall -y opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless

!pip install -q opencv-contrib-python==4.10.0.84

Found existing installation: opencv-python 5.0.0.93
Uninstalling opencv-python-5.0.0.93:
  Successfully uninstalled opencv-python-5.0.0.93
Found existing installation: opencv-python-headless 5.0.0.93
Uninstalling opencv-python-headless-5.0.0.93:
  Successfully uninstalled opencv-python-headless-5.0.0.93
Found existing installation: opencv-contrib-python 4.13.0.92
Uninstalling opencv-contrib-python-4.13.0.92:
  Successfully uninstalled opencv-contrib-python-4.13.0.92
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 8.7 MB/s eta 0:00:00


In [ ]:
import os
os.kill(os.getpid(), 9)

In [2]:

# ============================================================
# CELL 1 — DEPENDENCIES, IMPORTS, REPRODUCIBILITY
# ============================================================

!pip -q install "scikit-image>=0.24,<0.26" "scikit-learn>=1.5,<1.9" "PyYAML>=6,<7" "joblib>=1.4,<2"

import os
import re
import gc
import sys
import json
import math
import time
import yaml
import random
import shutil
import hashlib
import logging
import platform
import subprocess
import unicodedata
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from concurrent.futures import ThreadPoolExecutor

import cv2
import joblib
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import matplotlib.pyplot as plt

from skimage.feature import hog, local_binary_pattern

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

SEED = 42

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)

if not hasattr(cv2, "KAZE_create"):
    raise RuntimeError(
        "Bu OpenCV sürümünde cv2.KAZE_create bulunamadı. "
        "Colab runtime'ını güncel bir OpenCV sürümüyle yeniden başlatın."
    )

_kaze_probe = cv2.KAZE_create()

print("=" * 78)
print("ENVIRONMENT")
print("=" * 78)
print("Python       :", platform.python_version())
print("OpenCV       :", cv2.__version__)
print("NumPy        :", np.__version__)
print("Pandas       :", pd.__version__)
print("KAZE         : AVAILABLE")
print("Seed         :", SEED)


ENVIRONMENT
Python       : 3.12.13
OpenCV       : 4.10.0
NumPy        : 2.0.2
Pandas       : 2.2.2
KAZE         : AVAILABLE
Seed         : 42


In [3]:

# ============================================================
# CELL 2 — GOOGLE DRIVE + STRICT PROJECT PATH RESOLUTION
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

MYDRIVE = Path("/content/drive/MyDrive")

def _norm_name(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text.casefold().strip()

def find_unique_child(parent: Path, accepted_names):
    if not parent.is_dir():
        raise FileNotFoundError(f"Parent directory not found: {parent}")

    wanted = {_norm_name(name) for name in accepted_names}
    matches = [
        p for p in parent.iterdir()
        if p.is_dir() and _norm_name(p.name) in wanted
    ]

    if len(matches) == 1:
        return matches[0]

    if not matches:
        raise FileNotFoundError(
            f"Expected one of {accepted_names} under {parent}.\n"
            f"Existing folders: {[p.name for p in parent.iterdir() if p.is_dir()]}"
        )

    raise RuntimeError(f"Ambiguous folder match under {parent}: {matches}")

AISC_ROOT = find_unique_child(
    MYDRIVE,
    [
        "AISC DeepFake Çalışmaları",
        "AISC Deepfake Çalışmaları",
        "AISC Çalışmalar",
        "AISC Çalışmalar",
    ],
)
DENEYLER_ROOT = find_unique_child(AISC_ROOT, ["Deneyler"])
KADER_ROOT = find_unique_child(DENEYLER_ROOT, ["Kader"])
DENEY1_ROOT = find_unique_child(KADER_ROOT, ["Deney 1", "Deney1"])
EYE_ROOT = find_unique_child(DENEY1_ROOT, ["Göz", "Goz"])
DATA_ROOT = find_unique_child(EYE_ROOT, ["eye_roi_output"])
RESULTS_ROOT = find_unique_child(DENEY1_ROOT, ["Sonuçlar", "Sonuclar"])

ROI_METADATA_PATH = DATA_ROOT / "metadata.csv"
if not ROI_METADATA_PATH.is_file():
    raise FileNotFoundError(f"Eye ROI metadata not found: {ROI_METADATA_PATH}")

print("=" * 78)
print("RESOLVED PATHS")
print("=" * 78)
print("Experiment root :", DENEY1_ROOT)
print("Eye ROI root    :", DATA_ROOT)
print("Metadata        :", ROI_METADATA_PATH)
print("Results root    :", RESULTS_ROOT)


Mounted at /content/drive
RESOLVED PATHS
Experiment root : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1
Eye ROI root    : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output
Metadata        : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Göz/eye_roi_output/metadata.csv
Results root    : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar


In [4]:

# ============================================================
# CELL 3 — CONFIG + RUN ID + OUTPUT DIRECTORIES
# ============================================================

DEFAULT_CONFIG = {
    "seed": 42,
    "region": "eye",
    "roi_variant": "combined_eye",
    "image_size": [224, 224],

    "hog": {
        "orientations": 9,
        "pixels_per_cell": [16, 16],
        "cells_per_block": [2, 2],
        "block_norm": "L2-Hys",
        "transform_sqrt": True,
    },

    "lbp": {
        "radius": 3,
        "points": 24,
        "method": "uniform",
    },

    "kaze": {
        "descriptor_length": 64,
    },

    "svm": {
        "C_values": [0.1, 1.0, 10.0],
        "kernel": "rbf",
        "gamma": "scale",
        "class_weight": "balanced",
        "cache_size_mb": 2000,
    },

    "random_forest_candidates": [
        {
            "name": "RF-1",
            "n_estimators": 300,
            "max_depth": None,
            "min_samples_leaf": 1,
            "max_features": "sqrt",
        },
        {
            "name": "RF-2",
            "n_estimators": 300,
            "max_depth": 20,
            "min_samples_leaf": 1,
            "max_features": "sqrt",
        },
        {
            "name": "RF-3",
            "n_estimators": 300,
            "max_depth": None,
            "min_samples_leaf": 2,
            "max_features": "sqrt",
        },
    ],

    "selection_metric": "f1",
    "positive_class": "fake",
    "compute_sha256": True,
    "sha256_workers": 8,
    "verify_images": True,
    "feature_cache": True,
    "figure_dpi": 150,
    "min_figure_short_edge_px": 600,
}

CONFIG_DIR = DENEY1_ROOT / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
MASTER_CONFIG_PATH = CONFIG_DIR / "experiment_eye_hog_lbp_kaze_svm_rf.yaml"

if not MASTER_CONFIG_PATH.exists():
    tmp = MASTER_CONFIG_PATH.with_suffix(".yaml.tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        yaml.safe_dump(DEFAULT_CONFIG, f, sort_keys=False, allow_unicode=True)
    os.replace(tmp, MASTER_CONFIG_PATH)

try:
    with open(MASTER_CONFIG_PATH, "r", encoding="utf-8") as f:
        CONFIG = yaml.safe_load(f)
except yaml.YAMLError as exc:
    broken = MASTER_CONFIG_PATH.with_suffix(".yaml.broken")
    shutil.copy2(MASTER_CONFIG_PATH, broken)
    raise RuntimeError(
        f"YAML config bozuk. Yedek oluşturuldu: {broken}"
    ) from exc

missing_config = set(DEFAULT_CONFIG) - set(CONFIG)
if missing_config:
    raise KeyError(f"Config missing keys: {sorted(missing_config)}")

SEED = int(CONFIG["seed"])
seed_everything(SEED)

base_run_id = (
    datetime.now(ZoneInfo("Europe/Istanbul"))
    .strftime("%Y%m%d_%H%M")
    + f"_eye_hog_lbp_kaze_svm_rf_seed{SEED}"
)

RUN_ID = base_run_id
run_index = 2
while (RESULTS_ROOT / RUN_ID).exists():
    RUN_ID = f"{base_run_id}_r{run_index}"
    run_index += 1

RUN_DIR = RESULTS_ROOT / RUN_ID
DIRS = {
    "checkpoints": RUN_DIR / "checkpoints",
    "logs": RUN_DIR / "logs",
    "metrics": RUN_DIR / "metrics",
    "predictions": RUN_DIR / "predictions",
    "figures": RUN_DIR / "figures",
    "artifacts": RUN_DIR / "artifacts",
}
for d in [RUN_DIR, *DIRS.values()]:
    d.mkdir(parents=True, exist_ok=False if d == RUN_DIR else True)

RESOLVED_CONFIG = RUN_DIR / "config_resolved.yaml"
with open(RESOLVED_CONFIG.with_suffix(".yaml.tmp"), "w", encoding="utf-8") as f:
    yaml.safe_dump(CONFIG, f, sort_keys=False, allow_unicode=True)
os.replace(RESOLVED_CONFIG.with_suffix(".yaml.tmp"), RESOLVED_CONFIG)

print("Run ID      :", RUN_ID)
print("Run directory:", RUN_DIR)


Run ID      : 20260807_1719_eye_hog_lbp_kaze_svm_rf_seed42
Run directory: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260807_1719_eye_hog_lbp_kaze_svm_rf_seed42


In [5]:

# ============================================================
# CELL 4 — LOGGING, ATOMIC I/O, HASHING, ENVIRONMENT LOCK
# ============================================================

LOG_FILE = DIRS["logs"] / "run.log"

logger = logging.getLogger("eye_hog_lbp_kaze_svm_rf")
logger.handlers.clear()
logger.setLevel(logging.INFO)

_formatter = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s"
)
_file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
_file_handler.setFormatter(_formatter)
_stream_handler = logging.StreamHandler()
_stream_handler.setFormatter(_formatter)
logger.addHandler(_file_handler)
logger.addHandler(_stream_handler)

def write_json_atomic(payload, target: Path):
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(target.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    with open(tmp, "r", encoding="utf-8") as f:
        json.load(f)
    os.replace(tmp, target)

def write_csv_atomic(df: pd.DataFrame, target: Path):
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(target.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    check = pd.read_csv(tmp)
    if len(check) != len(df):
        raise RuntimeError(f"CSV integrity check failed: {target}")
    os.replace(tmp, target)

def write_joblib_atomic(payload, target: Path):
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(target.suffix + ".tmp")
    joblib.dump(payload, tmp)
    _ = joblib.load(tmp)
    os.replace(tmp, target)

def write_npz_atomic(target: Path, **arrays):
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(".tmp.npz")
    np.savez_compressed(tmp, **arrays)
    with np.load(tmp, allow_pickle=False) as check:
        for key in arrays:
            if key not in check:
                raise RuntimeError(f"NPZ integrity failure: {key}")
    os.replace(tmp, target)

def sha256_file(path: Path, block_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            h.update(block)
    return h.hexdigest()

def stable_audit_id(index, row):
    payload = (
        f"{index}|{row.get('label','')}|{row.get('split','')}|"
        f"{row.get('source_frame','')}|{row.get('frame_stem','')}|"
        f"{row.get('status_raw','')}"
    )
    return "audit_" + hashlib.sha256(payload.encode("utf-8")).hexdigest()[:20]

environment = {
    "python": platform.python_version(),
    "opencv": cv2.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "seed": SEED,
}
write_json_atomic(environment, RUN_DIR / "environment.json")

freeze = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"],
    text=True,
)
tmp = RUN_DIR / "requirements_lock.txt.tmp"
tmp.write_text(freeze, encoding="utf-8")
os.replace(tmp, RUN_DIR / "requirements_lock.txt")

logger.info("Atomic I/O and environment lock ready.")


2026-08-07 14:20:02,362 | INFO | Atomic I/O and environment lock ready.
INFO:eye_hog_lbp_kaze_svm_rf:Atomic I/O and environment lock ready.



## Cell 5 — Eye ROI Metadata Quality Gate

Bu hücre, daha önce göz ROI üretiminde oluşturulan `metadata.csv` dosyasını giriş olarak kullanır. `no_face` kayıtları model eğitimine dahil edilmez. `SUCCESS` kayıtlarındaki gerçek `sample_id` değerleri benzersiz olmak zorundadır; `SKIPPED` satırlardaki boş kimlikler yalnız denetim için deterministic audit ID alır.

Aynı görüntü içeriğinin farklı splitlerde bulunup bulunmadığı SHA256 ile kontrol edilir. Mevcut `video_id` alanı gerçek orijinal video kimliğini kanıtlamıyorsa video-level leakage sonucu uydurulmaz.


In [6]:

# ============================================================
# CELL 5 — EYE ROI METADATA + DATA ACCOUNTING + LEAKAGE GATES
# ============================================================

roi = pd.read_csv(ROI_METADATA_PATH)

required_roi_cols = {
    "sample_id",
    "source_frame",
    "relative_frame_path",
    "label",
    "split",
    "video_id",
    "frame_stem",
    "face_id",
    "combined_eye_path",
    "status",
}
missing = required_roi_cols - set(roi.columns)
if missing:
    raise AssertionError(f"Eye ROI metadata missing columns: {sorted(missing)}")

roi = roi.copy()

def norm_string(series):
    return series.fillna("").astype(str).str.strip()

roi["sample_id"] = norm_string(roi["sample_id"])
roi["source_frame"] = norm_string(roi["source_frame"])
roi["relative_frame_path"] = norm_string(roi["relative_frame_path"])
roi["label"] = norm_string(roi["label"]).str.lower()
roi["split"] = norm_string(roi["split"]).str.lower().replace(
    {"validation": "val", "valid": "val"}
)
roi["status_raw"] = norm_string(roi["status"]).str.lower()
roi["video_id"] = norm_string(roi["video_id"])
roi["frame_stem"] = norm_string(roi["frame_stem"])

if set(roi["label"].unique()) != {"real", "fake"}:
    raise AssertionError(roi["label"].value_counts(dropna=False).to_dict())
if set(roi["split"].unique()) != {"train", "val", "test"}:
    raise AssertionError(roi["split"].value_counts(dropna=False).to_dict())

status_map = {
    "ok": "SUCCESS",
    "success": "SUCCESS",
    "no_face": "SKIPPED",
    "skipped": "SKIPPED",
    "error": "ERROR",
}
unknown_statuses = set(roi["status_raw"].unique()) - set(status_map)
if unknown_statuses:
    raise AssertionError(f"Unknown statuses: {sorted(unknown_statuses)}")

roi["status_std"] = roi["status_raw"].map(status_map)
success_roi_mask = roi["status_std"].eq("SUCCESS")

if roi.loc[success_roi_mask, "sample_id"].eq("").any():
    raise AssertionError("SUCCESS Eye ROI rows contain empty sample_id.")

dup_success = roi.loc[success_roi_mask, "sample_id"].duplicated(keep=False)
if dup_success.any():
    bad_ids = roi.loc[success_roi_mask].loc[dup_success, "sample_id"].unique()[:20]
    raise AssertionError(f"Duplicate SUCCESS sample_id values: {bad_ids.tolist()}")

missing_id_mask = roi["sample_id"].eq("")
generated_audit_ids = 0
for idx in roi.index[missing_id_mask]:
    roi.at[idx, "sample_id"] = stable_audit_id(idx, roi.loc[idx])
    generated_audit_ids += 1

if not roi["sample_id"].is_unique:
    raise AssertionError("Metadata trace IDs are not unique after audit ID generation.")

def resolve_eye_path(value):
    if pd.isna(value) or str(value).strip() == "":
        return ""
    p = Path(str(value).strip())
    return str(p if p.is_absolute() else DATA_ROOT / p)

def parse_frame_index(frame_stem):
    match = re.search(r"(\d+)$", str(frame_stem))
    return int(match.group(1)) if match else -1

def parse_face_index(value):
    if pd.isna(value) or str(value).strip() == "":
        return -1
    try:
        return int(float(value))
    except Exception:
        return -1

metadata = pd.DataFrame({
    "sample_id": roi["sample_id"],
    "sample_id_semantics": np.where(
        success_roi_mask, "roi_sample_id", "generated_audit_id"
    ),
    "source_video": roi["video_id"],
    "source_video_semantics": "roi_metadata_video_id",
    "source_frame": roi["source_frame"],
    "frame_index": roi["frame_stem"].map(parse_frame_index).astype(int),
    "face_index": roi["face_id"].map(parse_face_index).astype(int),
    "roi_state": "combined_eye",
    "label": roi["label"],
    "split": roi["split"],
    "status": roi["status_std"],
    "skip_reason": np.where(
        roi["status_std"].eq("SKIPPED"),
        roi["status_raw"],
        np.where(
            roi["status_std"].eq("ERROR"),
            (
                roi["error"].fillna("").astype(str)
                if "error" in roi.columns
                else ""
            ),
            "",
        ),
    ),
    "sha256": "",
    "output_path": roi["combined_eye_path"].map(resolve_eye_path),
    "relative_frame_path": roi["relative_frame_path"],
    "frame_stem": roi["frame_stem"],
    "video_id": roi["video_id"],
    "run_id": RUN_ID,
})

success_mask = metadata["status"].eq("SUCCESS")
if not metadata.loc[success_mask, "output_path"].ne("").all():
    raise AssertionError("Some SUCCESS rows have empty combined_eye_path.")

total_inputs = len(metadata)
success_count = int(success_mask.sum())
skipped_count = int(metadata["status"].eq("SKIPPED").sum())
error_count = int(metadata["status"].eq("ERROR").sum())

if total_inputs != success_count + skipped_count + error_count:
    raise AssertionError("Input accounting mismatch.")

missing_files = [
    p for p in metadata.loc[success_mask, "output_path"]
    if not Path(p).is_file()
]
if missing_files:
    raise FileNotFoundError(
        f"{len(missing_files)} SUCCESS combined-eye files are missing. "
        f"Examples: {missing_files[:10]}"
    )

eligible = metadata.loc[success_mask].copy().reset_index(drop=True)
if eligible.empty:
    raise AssertionError("No SUCCESS Eye ROI samples.")

if not eligible["frame_index"].ge(0).all():
    raise AssertionError("Unresolved frame_index among SUCCESS samples.")
if not eligible["face_index"].ge(0).all():
    raise AssertionError("Unresolved face_index among SUCCESS samples.")

for split_name in ["train", "val", "test"]:
    labels = set(eligible.loc[eligible["split"].eq(split_name), "label"])
    if labels != {"real", "fake"}:
        raise AssertionError(f"{split_name} does not contain both classes: {labels}")

# Exact-path split leakage.
path_split_counts = eligible.groupby("output_path")["split"].nunique()
cross_split_path_count = int((path_split_counts > 1).sum())
if cross_split_path_count:
    raise AssertionError(
        f"Same combined-eye path appears in multiple splits: "
        f"{cross_split_path_count}"
    )

# SHA256 exact-content leakage.
cross_split_hash_count = 0
if bool(CONFIG["compute_sha256"]):
    logger.info("Computing SHA256 for %d SUCCESS images.", len(eligible))

    workers = max(1, int(CONFIG["sha256_workers"]))
    paths = [Path(p) for p in eligible["output_path"]]

    with ThreadPoolExecutor(max_workers=workers) as executor:
        hashes = list(
            tqdm(
                executor.map(sha256_file, paths),
                total=len(paths),
                desc="SHA256",
            )
        )

    eligible["sha256"] = hashes
    sha_map = eligible.set_index("sample_id")["sha256"].to_dict()
    metadata.loc[success_mask, "sha256"] = (
        metadata.loc[success_mask, "sample_id"].map(sha_map)
    )

    hash_split_counts = eligible.groupby("sha256")["split"].nunique()
    cross_split_hash_count = int((hash_split_counts > 1).sum())
    if cross_split_hash_count:
        raise AssertionError(
            f"Exact same image content crosses splits: "
            f"{cross_split_hash_count} SHA256 values."
        )

# Current metadata video_id group check.
group_sets = {
    split: set(
        eligible.loc[eligible["split"].eq(split), "video_id"]
        .astype(str)
        .str.strip()
    )
    for split in ["train", "val", "test"]
}
group_intersections = {
    "train_val": len(group_sets["train"] & group_sets["val"]),
    "train_test": len(group_sets["train"] & group_sets["test"]),
    "val_test": len(group_sets["val"] & group_sets["test"]),
}
if any(v != 0 for v in group_intersections.values()):
    raise AssertionError(f"Current video_id group leakage: {group_intersections}")

if eligible.groupby("video_id")["split"].nunique().max() > 1:
    raise AssertionError("A current metadata video_id crosses splits.")
if eligible.groupby("video_id")["label"].nunique().max() > 1:
    raise AssertionError("A current metadata video_id crosses labels.")

split_class_counts_df = (
    eligible.groupby(["split", "label"]).size().unstack(fill_value=0)
)
split_class_counts = split_class_counts_df.to_dict(orient="index")
split_group_counts = {
    split: int(
        eligible.loc[eligible["split"].eq(split), "video_id"].nunique()
    )
    for split in ["train", "val", "test"]
}

true_video_status = "NOT_VERIFIABLE_FROM_CURRENT_METADATA"

accounting = {
    "run_id": RUN_ID,
    "total_inputs": int(total_inputs),
    "success_count": success_count,
    "skipped_count": skipped_count,
    "error_count": error_count,
    "generated_audit_sample_ids": generated_audit_ids,
    "training_eligible_success_count": int(len(eligible)),
    "split_class_counts": split_class_counts,
    "video_id_group_counts": split_group_counts,
    "video_id_group_intersections": group_intersections,
    "cross_split_duplicate_output_path_count": cross_split_path_count,
    "cross_split_duplicate_sha256_count": cross_split_hash_count,
    "true_video_level_leakage_status": true_video_status,
    "true_video_level_note": (
        "ROI metadata video_id is preserved exactly as provided. "
        "It is not treated as authoritative original source-video identity."
    ),
}

write_json_atomic(accounting, DIRS["metrics"] / "data_accounting.json")
write_csv_atomic(metadata, DIRS["artifacts"] / "metadata_used.csv")

print("=" * 78)
print("EYE ROI DATA ACCOUNTING")
print("=" * 78)
print(json.dumps(accounting, ensure_ascii=False, indent=2))


2026-08-07 14:20:30,981 | INFO | Computing SHA256 for 2986 SUCCESS images.
INFO:eye_hog_lbp_kaze_svm_rf:Computing SHA256 for 2986 SUCCESS images.


SHA256:   0%|          | 0/2986 [00:00<?, ?it/s]

EYE ROI DATA ACCOUNTING
{
  "run_id": "20260807_1719_eye_hog_lbp_kaze_svm_rf_seed42",
  "total_inputs": 3097,
  "success_count": 2986,
  "skipped_count": 111,
  "error_count": 0,
  "generated_audit_sample_ids": 111,
  "training_eligible_success_count": 2986,
  "split_class_counts": {
    "test": {
      "fake": 156,
      "real": 146
    },
    "train": {
      "fake": 1191,
      "real": 1197
    },
    "val": {
      "fake": 141,
      "real": 155
    }
  },
  "video_id_group_counts": {
    "train": 2,
    "val": 2,
    "test": 2
  },
  "video_id_group_intersections": {
    "train_val": 0,
    "train_test": 0,
    "val_test": 0
  },
  "cross_split_duplicate_output_path_count": 0,
  "cross_split_duplicate_sha256_count": 0,
  "true_video_level_leakage_status": "NOT_VERIFIABLE_FROM_CURRENT_METADATA",
  "true_video_level_note": "ROI metadata video_id is preserved exactly as provided. It is not treated as authoritative original source-video identity."
}



## Cell 6 — HOG, LBP ve KAZE Özellikleri

Orijinal ağız deneyindeki feature tasarımı korunmuştur; yalnızca yorumlar ve veri kaynağı göz bölgesine uyarlanmıştır.

- **HOG:** Göz kapağı, kirpik, göz köşesi ve lokal kenar/yönelim bilgisi.
- **LBP:** Göz çevresi deri dokusu ve mikro-kontrast örüntüleri.
- **KAZE:** Nonlinear scale-space içindeki lokal keypoint ve descriptor bilgisi.
- KAZE keypoint sayısı değişken olduğundan descriptor'ların **ortalaması + standart sapması** kullanılarak sabit 128 boyutlu vektör üretilir.


In [7]:

# ============================================================
# CELL 6 — FEATURE EXTRACTION FUNCTIONS
# ============================================================

IMAGE_SIZE = tuple(int(v) for v in CONFIG["image_size"])

HOG_ORIENTATIONS = int(CONFIG["hog"]["orientations"])
HOG_PIXELS_PER_CELL = tuple(int(v) for v in CONFIG["hog"]["pixels_per_cell"])
HOG_CELLS_PER_BLOCK = tuple(int(v) for v in CONFIG["hog"]["cells_per_block"])
HOG_BLOCK_NORM = str(CONFIG["hog"]["block_norm"])
HOG_TRANSFORM_SQRT = bool(CONFIG["hog"]["transform_sqrt"])

LBP_RADIUS = int(CONFIG["lbp"]["radius"])
LBP_POINTS = int(CONFIG["lbp"]["points"])
LBP_METHOD = str(CONFIG["lbp"]["method"])
LBP_BINS = LBP_POINTS + 2

KAZE_DESCRIPTOR_LENGTH = int(CONFIG["kaze"]["descriptor_length"])

def load_and_preprocess_image(image_path):
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f"Image cannot be read: {image_path}")

    image = cv2.resize(
        image,
        IMAGE_SIZE,
        interpolation=cv2.INTER_AREA,
    )
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    if gray.dtype != np.uint8:
        gray = np.clip(gray, 0, 255).astype(np.uint8)

    return gray

def extract_hog_features(gray):
    vector = hog(
        gray,
        orientations=HOG_ORIENTATIONS,
        pixels_per_cell=HOG_PIXELS_PER_CELL,
        cells_per_block=HOG_CELLS_PER_BLOCK,
        block_norm=HOG_BLOCK_NORM,
        transform_sqrt=HOG_TRANSFORM_SQRT,
        feature_vector=True,
    )
    return np.asarray(vector, dtype=np.float32)

def extract_lbp_features(gray):
    lbp_img = local_binary_pattern(
        gray,
        P=LBP_POINTS,
        R=LBP_RADIUS,
        method=LBP_METHOD,
    )
    hist, _ = np.histogram(
        lbp_img.ravel(),
        bins=np.arange(0, LBP_BINS + 1),
        range=(0, LBP_BINS),
    )
    hist = hist.astype(np.float32)
    hist /= hist.sum() + 1e-8
    return hist

def extract_kaze_features(gray):
    # New detector per call avoids shared mutable detector state.
    detector = cv2.KAZE_create()
    keypoints, descriptors = detector.detectAndCompute(gray, None)

    keypoint_count = 0 if keypoints is None else len(keypoints)

    if descriptors is None or len(descriptors) == 0:
        return (
            np.zeros(KAZE_DESCRIPTOR_LENGTH * 2, dtype=np.float32),
            0,
        )

    descriptors = np.asarray(descriptors, dtype=np.float32)

    if descriptors.ndim != 2:
        raise ValueError(f"Unexpected KAZE descriptor shape: {descriptors.shape}")

    if descriptors.shape[1] != KAZE_DESCRIPTOR_LENGTH:
        raise ValueError(
            f"Unexpected KAZE descriptor length: "
            f"{descriptors.shape[1]} != {KAZE_DESCRIPTOR_LENGTH}"
        )

    vector = np.concatenate(
        [
            descriptors.mean(axis=0),
            descriptors.std(axis=0),
        ]
    ).astype(np.float32)

    return vector, keypoint_count

def extract_feature_components(image_path):
    gray = load_and_preprocess_image(image_path)

    hog_vector = extract_hog_features(gray)
    lbp_vector = extract_lbp_features(gray)
    kaze_vector, keypoint_count = extract_kaze_features(gray)

    base_vector = np.concatenate(
        [hog_vector, lbp_vector]
    ).astype(np.float32)

    fused_vector = np.concatenate(
        [hog_vector, lbp_vector, kaze_vector]
    ).astype(np.float32)

    for name, vec in {
        "HOG": hog_vector,
        "LBP": lbp_vector,
        "KAZE": kaze_vector,
        "HOG+LBP": base_vector,
        "HOG+LBP+KAZE": fused_vector,
    }.items():
        if not np.isfinite(vec).all():
            raise FloatingPointError(f"{name} contains NaN/Inf: {image_path}")

    info = {
        "hog_dim": int(len(hog_vector)),
        "lbp_dim": int(len(lbp_vector)),
        "kaze_dim": int(len(kaze_vector)),
        "base_dim": int(len(base_vector)),
        "fused_dim": int(len(fused_vector)),
        "kaze_keypoints": int(keypoint_count),
    }
    return base_vector, fused_vector, info

print("Feature extraction functions ready.")


Feature extraction functions ready.


In [8]:

# ============================================================
# CELL 7 — CLASSICAL ML SMOKE TEST
# ============================================================

train_meta = eligible.loc[eligible["split"].eq("train")].copy()
val_meta = eligible.loc[eligible["split"].eq("val")].copy()
test_meta = eligible.loc[eligible["split"].eq("test")].copy()

for split_name, df in [
    ("train", train_meta),
    ("val", val_meta),
    ("test", test_meta),
]:
    if df.empty:
        raise AssertionError(f"{split_name} split is empty.")
    if set(df["label"]) != {"real", "fake"}:
        raise AssertionError(f"{split_name} lacks both classes.")

smoke_rows = pd.concat(
    [
        train_meta.loc[train_meta["label"].eq("real")].head(2),
        train_meta.loc[train_meta["label"].eq("fake")].head(2),
    ],
    ignore_index=True,
)
if len(smoke_rows) != 4:
    raise AssertionError("Smoke test requires 2 REAL + 2 FAKE train samples.")

smoke_base = []
smoke_fused = []
smoke_y = []

smoke_info = None

for _, row in smoke_rows.iterrows():
    base_vec, fused_vec, info = extract_feature_components(
        row["output_path"]
    )
    smoke_base.append(base_vec)
    smoke_fused.append(fused_vec)
    smoke_y.append(1 if row["label"] == "fake" else 0)

    if smoke_info is None:
        smoke_info = info
    else:
        for key in ["hog_dim", "lbp_dim", "kaze_dim", "base_dim", "fused_dim"]:
            if info[key] != smoke_info[key]:
                raise AssertionError(
                    f"Feature dimension mismatch for {key}: "
                    f"{info[key]} != {smoke_info[key]}"
                )

X_smoke = np.vstack(smoke_base).astype(np.float32)
y_smoke = np.asarray(smoke_y, dtype=np.int64)

smoke_scaler = StandardScaler()
X_smoke_scaled = smoke_scaler.fit_transform(X_smoke)

smoke_svm = SVC(
    kernel="linear",
    C=0.1,
    class_weight="balanced",
    random_state=SEED,
)
smoke_svm.fit(X_smoke_scaled, y_smoke)
smoke_pred = smoke_svm.predict(X_smoke_scaled)

if smoke_pred.shape != y_smoke.shape:
    raise AssertionError("Smoke inference shape mismatch.")

feature_dimensions = {
    **{k: int(v) for k, v in smoke_info.items() if k != "kaze_keypoints"},
    "image_size": list(IMAGE_SIZE),
}
write_json_atomic(
    feature_dimensions,
    DIRS["artifacts"] / "feature_dimensions.json",
)

print("=" * 78)
print("CLASSICAL ML SMOKE TEST PASSED")
print("=" * 78)
print(json.dumps(feature_dimensions, indent=2))


CLASSICAL ML SMOKE TEST PASSED
{
  "hog_dim": 6084,
  "lbp_dim": 26,
  "kaze_dim": 128,
  "base_dim": 6110,
  "fused_dim": 6238,
  "image_size": [
    224,
    224
  ]
}



## Cell 8 — Tüm Veriden Feature Extraction + Cache

Train, validation ve test için HOG+LBP ve HOG+LBP+KAZE özellikleri çıkarılır. Sonuçlar NPZ cache dosyalarına atomik kaydedilir. Hücre yeniden çalıştırılırsa geçerli cache yeniden hesaplanmaz.


In [9]:

# ============================================================
# CELL 8 — EXTRACT / LOAD FEATURE CACHE
# ============================================================

def feature_cache_path(split_name):
    return DIRS["artifacts"] / f"features_{split_name}.npz"

def metadata_for_split(split_name):
    return (
        eligible.loc[eligible["split"].eq(split_name)]
        .sort_values(["label", "sample_id"])
        .reset_index(drop=True)
    )

def extract_split_features(split_name):
    split_df = metadata_for_split(split_name)
    cache_path = feature_cache_path(split_name)

    if bool(CONFIG["feature_cache"]) and cache_path.exists():
        logger.info("Loading feature cache: %s", cache_path)
        with np.load(cache_path, allow_pickle=False) as data:
            X_base = data["X_base"]
            X_fused = data["X_fused"]
            y = data["y"]
            sample_ids = data["sample_ids"].astype(str)
            kaze_counts = data["kaze_keypoints"]

        expected_ids = split_df["sample_id"].astype(str).to_numpy()
        if not np.array_equal(sample_ids, expected_ids):
            raise RuntimeError(
                f"Feature cache sample IDs do not match current metadata: "
                f"{split_name}"
            )

        return split_df, X_base, X_fused, y, kaze_counts

    base_rows = []
    fused_rows = []
    labels = []
    kaze_counts = []

    start = time.time()

    for _, row in tqdm(
        split_df.iterrows(),
        total=len(split_df),
        desc=f"Features {split_name}",
    ):
        base_vec, fused_vec, info = extract_feature_components(
            row["output_path"]
        )
        base_rows.append(base_vec)
        fused_rows.append(fused_vec)
        labels.append(1 if row["label"] == "fake" else 0)
        kaze_counts.append(info["kaze_keypoints"])

    X_base = np.vstack(base_rows).astype(np.float32)
    X_fused = np.vstack(fused_rows).astype(np.float32)
    y = np.asarray(labels, dtype=np.int64)
    kaze_counts = np.asarray(kaze_counts, dtype=np.int32)
    sample_ids = split_df["sample_id"].astype(str).to_numpy(dtype=str)

    if not np.isfinite(X_base).all():
        raise FloatingPointError(f"Non-finite HOG+LBP features in {split_name}.")
    if not np.isfinite(X_fused).all():
        raise FloatingPointError(
            f"Non-finite HOG+LBP+KAZE features in {split_name}."
        )

    if len(X_base) != len(split_df) or len(X_fused) != len(split_df):
        raise RuntimeError(f"Feature row accounting mismatch in {split_name}.")

    write_npz_atomic(
        cache_path,
        X_base=X_base,
        X_fused=X_fused,
        y=y,
        sample_ids=sample_ids,
        kaze_keypoints=kaze_counts,
    )

    logger.info(
        "%s feature extraction completed in %.2f seconds.",
        split_name,
        time.time() - start,
    )

    return split_df, X_base, X_fused, y, kaze_counts

(
    train_meta,
    X_train_base,
    X_train_fused,
    y_train,
    train_kaze_counts,
) = extract_split_features("train")

(
    val_meta,
    X_val_base,
    X_val_fused,
    y_val,
    val_kaze_counts,
) = extract_split_features("val")

(
    test_meta,
    X_test_base,
    X_test_fused,
    y_test,
    test_kaze_counts,
) = extract_split_features("test")

print("Train base/fused:", X_train_base.shape, X_train_fused.shape)
print("Val   base/fused:", X_val_base.shape, X_val_fused.shape)
print("Test  base/fused:", X_test_base.shape, X_test_fused.shape)


Features train:   0%|          | 0/2388 [00:00<?, ?it/s]

2026-08-07 14:32:04,881 | INFO | train feature extraction completed in 318.61 seconds.
INFO:eye_hog_lbp_kaze_svm_rf:train feature extraction completed in 318.61 seconds.


Features val:   0%|          | 0/296 [00:00<?, ?it/s]

2026-08-07 14:32:40,386 | INFO | val feature extraction completed in 35.48 seconds.
INFO:eye_hog_lbp_kaze_svm_rf:val feature extraction completed in 35.48 seconds.


Features test:   0%|          | 0/302 [00:00<?, ?it/s]

2026-08-07 14:33:16,108 | INFO | test feature extraction completed in 35.71 seconds.
INFO:eye_hog_lbp_kaze_svm_rf:test feature extraction completed in 35.71 seconds.


Train base/fused: (2388, 6110) (2388, 6238)
Val   base/fused: (296, 6110) (296, 6238)
Test  base/fused: (302, 6110) (302, 6238)


In [10]:

# ============================================================
# CELL 9 — TRAIN-ONLY SCALING + METRIC UTILITIES
# ============================================================

base_scaler = StandardScaler()
fused_scaler = StandardScaler()

X_train_base_scaled = base_scaler.fit_transform(X_train_base)
X_val_base_scaled = base_scaler.transform(X_val_base)

X_train_fused_scaled = fused_scaler.fit_transform(X_train_fused)
X_val_fused_scaled = fused_scaler.transform(X_val_fused)

# Test is transformed only after all validation-side model selection is complete.
if not np.isfinite(X_train_base_scaled).all():
    raise FloatingPointError("Scaled train base features contain NaN/Inf.")
if not np.isfinite(X_train_fused_scaled).all():
    raise FloatingPointError("Scaled train fused features contain NaN/Inf.")

def metrics_from_scores(y_true, scores, threshold):
    y_true = np.asarray(y_true, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    pred = (scores >= float(threshold)).astype(np.int64)

    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    result = {
        "accuracy": float(accuracy_score(y_true, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "specificity": float(tn / (tn + fp)) if (tn + fp) else 0.0,
        "roc_auc": float(roc_auc_score(y_true, scores)),
        "average_precision": float(average_precision_score(y_true, scores)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }
    return result, pred

def best_f1_threshold(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)

    # thresholds has len = len(precision)-1.
    if len(thresholds) == 0:
        raise RuntimeError("Threshold search returned no thresholds.")

    f1_values = (
        2 * precision[:-1] * recall[:-1]
        / (precision[:-1] + recall[:-1] + 1e-12)
    )
    best_index = int(np.nanargmax(f1_values))
    threshold = float(thresholds[best_index])

    metrics, _ = metrics_from_scores(y_true, scores, threshold)
    return threshold, metrics

print("Train-only scaling complete.")


Train-only scaling complete.



## Cell 10 — Validation Üzerinde Model Seçimi

Bu hücre üç model ailesini **yalnız train + validation** kullanarak finalize eder:

- HOG+LBP RBF-SVM: `C ∈ {0.1, 1, 10}`
- HOG+LBP+KAZE RBF-SVM: `C ∈ {0.1, 1, 10}`
- HOG+LBP+KAZE Random Forest: üç önceden tanımlı yapı

Her aday için karar threshold'u validation F1'i maksimize edecek biçimde bulunur. Test setine bu aşamada erişilmez.


In [11]:

# ============================================================
# CELL 10 — VALIDATION-ONLY MODEL + THRESHOLD SELECTION
# ============================================================

C_VALUES = [float(v) for v in CONFIG["svm"]["C_values"]]

def select_svm_family(
    family_name,
    X_train,
    X_val,
    y_train,
    y_val,
):
    rows = []
    best = None

    for candidate_index, C in enumerate(C_VALUES, start=1):
        start = time.time()

        model = SVC(
            C=C,
            kernel=str(CONFIG["svm"]["kernel"]),
            gamma=CONFIG["svm"]["gamma"],
            class_weight=CONFIG["svm"]["class_weight"],
            probability=False,
            cache_size=float(CONFIG["svm"]["cache_size_mb"]),
            random_state=SEED,
        )
        model.fit(X_train, y_train)

        val_scores = model.decision_function(X_val)
        threshold, val_metrics = best_f1_threshold(y_val, val_scores)

        record = {
            "family": family_name,
            "candidate": candidate_index,
            "C": C,
            "gamma": str(CONFIG["svm"]["gamma"]),
            "threshold": threshold,
            **{f"val_{k}": v for k, v in val_metrics.items()},
            "seconds": float(time.time() - start),
        }
        rows.append(record)

        score = (val_metrics["f1"], val_metrics["roc_auc"])
        if best is None or score > best["score"]:
            best = {
                "score": score,
                "model": model,
                "C": C,
                "threshold": threshold,
                "validation_metrics": val_metrics,
            }

        logger.info(
            "%s candidate %d/%d | C=%s | val_f1=%.4f | val_auc=%.4f",
            family_name,
            candidate_index,
            len(C_VALUES),
            C,
            val_metrics["f1"],
            val_metrics["roc_auc"],
        )

    return best, pd.DataFrame(rows)

baseline_best, baseline_search = select_svm_family(
    "HOG + LBP -> RBF-SVM",
    X_train_base_scaled,
    X_val_base_scaled,
    y_train,
    y_val,
)

fused_best, fused_search = select_svm_family(
    "HOG + LBP + KAZE -> RBF-SVM",
    X_train_fused_scaled,
    X_val_fused_scaled,
    y_train,
    y_val,
)

rf_rows = []
rf_best = None

for candidate_index, params in enumerate(
    CONFIG["random_forest_candidates"],
    start=1,
):
    start = time.time()

    model = RandomForestClassifier(
        n_estimators=int(params["n_estimators"]),
        max_depth=params["max_depth"],
        min_samples_leaf=int(params["min_samples_leaf"]),
        max_features=params["max_features"],
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    )
    model.fit(X_train_fused, y_train)

    val_scores = model.predict_proba(X_val_fused)[:, 1]
    threshold, val_metrics = best_f1_threshold(y_val, val_scores)

    record = {
        "family": "HOG + LBP + KAZE -> Random Forest",
        "candidate": candidate_index,
        "configuration": params["name"],
        "n_estimators": int(params["n_estimators"]),
        "max_depth": str(params["max_depth"]),
        "min_samples_leaf": int(params["min_samples_leaf"]),
        "max_features": params["max_features"],
        "threshold": threshold,
        **{f"val_{k}": v for k, v in val_metrics.items()},
        "seconds": float(time.time() - start),
    }
    rf_rows.append(record)

    score = (val_metrics["f1"], val_metrics["roc_auc"])
    if rf_best is None or score > rf_best["score"]:
        rf_best = {
            "score": score,
            "model": model,
            "configuration": params.copy(),
            "threshold": threshold,
            "validation_metrics": val_metrics,
        }

    logger.info(
        "RF candidate %d/%d | %s | val_f1=%.4f | val_auc=%.4f",
        candidate_index,
        len(CONFIG["random_forest_candidates"]),
        params["name"],
        val_metrics["f1"],
        val_metrics["roc_auc"],
    )

rf_search = pd.DataFrame(rf_rows)

all_validation_search = pd.concat(
    [baseline_search, fused_search, rf_search],
    ignore_index=True,
    sort=False,
)

write_csv_atomic(
    all_validation_search,
    DIRS["metrics"] / "validation_model_search.csv",
)

family_summary = pd.DataFrame([
    {
        "family": "HOG + LBP -> RBF-SVM",
        "selected_C": baseline_best["C"],
        "selected_configuration": "",
        "threshold": baseline_best["threshold"],
        **baseline_best["validation_metrics"],
    },
    {
        "family": "HOG + LBP + KAZE -> RBF-SVM",
        "selected_C": fused_best["C"],
        "selected_configuration": "",
        "threshold": fused_best["threshold"],
        **fused_best["validation_metrics"],
    },
    {
        "family": "HOG + LBP + KAZE -> Random Forest",
        "selected_C": np.nan,
        "selected_configuration": rf_best["configuration"]["name"],
        "threshold": rf_best["threshold"],
        **rf_best["validation_metrics"],
    },
])

write_csv_atomic(
    family_summary,
    DIRS["metrics"] / "validation_selected_families.csv",
)

print(family_summary)


2026-08-07 14:37:30,709 | INFO | HOG + LBP -> RBF-SVM candidate 1/3 | C=0.1 | val_f1=0.6879 | val_auc=0.6554
INFO:eye_hog_lbp_kaze_svm_rf:HOG + LBP -> RBF-SVM candidate 1/3 | C=0.1 | val_f1=0.6879 | val_auc=0.6554
2026-08-07 14:38:01,925 | INFO | HOG + LBP -> RBF-SVM candidate 2/3 | C=1.0 | val_f1=0.6852 | val_auc=0.6882
INFO:eye_hog_lbp_kaze_svm_rf:HOG + LBP -> RBF-SVM candidate 2/3 | C=1.0 | val_f1=0.6852 | val_auc=0.6882
2026-08-07 14:38:33,976 | INFO | HOG + LBP -> RBF-SVM candidate 3/3 | C=10.0 | val_f1=0.6699 | val_auc=0.6092
INFO:eye_hog_lbp_kaze_svm_rf:HOG + LBP -> RBF-SVM candidate 3/3 | C=10.0 | val_f1=0.6699 | val_auc=0.6092
2026-08-07 14:39:08,284 | INFO | HOG + LBP + KAZE -> RBF-SVM candidate 1/3 | C=0.1 | val_f1=0.6940 | val_auc=0.6599
INFO:eye_hog_lbp_kaze_svm_rf:HOG + LBP + KAZE -> RBF-SVM candidate 1/3 | C=0.1 | val_f1=0.6940 | val_auc=0.6599
2026-08-07 14:39:39,659 | INFO | HOG + LBP + KAZE -> RBF-SVM candidate 2/3 | C=1.0 | val_f1=0.6968 | val_auc=0.6814
INFO:eye_hog

                              family  selected_C selected_configuration  \
0               HOG + LBP -> RBF-SVM         0.1                          
1        HOG + LBP + KAZE -> RBF-SVM         1.0                          
2  HOG + LBP + KAZE -> Random Forest         NaN                   RF-2   

   threshold  accuracy  balanced_accuracy  precision    recall        f1  \
0  -0.377563  0.635135           0.644566   0.580488  0.843972  0.687861   
1  -0.635737  0.614865           0.629055   0.557447  0.929078  0.696809   
2   0.424118  0.594595           0.607779   0.545852  0.886525  0.675676   

   specificity   roc_auc  average_precision  tn   fp  fn   tp  
0     0.445161  0.655365           0.595690  69   86  22  119  
1     0.329032  0.681446           0.613573  51  104  10  131  
2     0.329032  0.660398           0.609624  51  104  16  125  


In [12]:

# ============================================================
# CELL 11 — ATOMIC MODEL PACKAGE + RELOAD QUALITY GATE
# ============================================================

model_packages = {
    "hog_lbp_rbf_svm": {
        "family": "HOG + LBP -> RBF-SVM",
        "model": baseline_best["model"],
        "scaler": base_scaler,
        "threshold": float(baseline_best["threshold"]),
        "validation_metrics": baseline_best["validation_metrics"],
        "config": CONFIG,
        "rng_state": {
            "python": random.getstate(),
            "numpy": np.random.get_state(),
        },
    },
    "hog_lbp_kaze_rbf_svm": {
        "family": "HOG + LBP + KAZE -> RBF-SVM",
        "model": fused_best["model"],
        "scaler": fused_scaler,
        "threshold": float(fused_best["threshold"]),
        "validation_metrics": fused_best["validation_metrics"],
        "config": CONFIG,
        "rng_state": {
            "python": random.getstate(),
            "numpy": np.random.get_state(),
        },
    },
    "hog_lbp_kaze_random_forest": {
        "family": "HOG + LBP + KAZE -> Random Forest",
        "model": rf_best["model"],
        "scaler": None,
        "threshold": float(rf_best["threshold"]),
        "validation_metrics": rf_best["validation_metrics"],
        "config": CONFIG,
        "rng_state": {
            "python": random.getstate(),
            "numpy": np.random.get_state(),
        },
    },
}

for name, package in model_packages.items():
    target = DIRS["checkpoints"] / f"{name}.joblib"
    write_joblib_atomic(package, target)

# Fresh-load score consistency on validation.
for name in model_packages:
    package = joblib.load(DIRS["checkpoints"] / f"{name}.joblib")

    if name == "hog_lbp_rbf_svm":
        X_val_input = package["scaler"].transform(X_val_base)
        original_scores = baseline_best["model"].decision_function(
            X_val_base_scaled
        )
        reloaded_scores = package["model"].decision_function(X_val_input)

    elif name == "hog_lbp_kaze_rbf_svm":
        X_val_input = package["scaler"].transform(X_val_fused)
        original_scores = fused_best["model"].decision_function(
            X_val_fused_scaled
        )
        reloaded_scores = package["model"].decision_function(X_val_input)

    else:
        original_scores = rf_best["model"].predict_proba(X_val_fused)[:, 1]
        reloaded_scores = package["model"].predict_proba(X_val_fused)[:, 1]

    if not np.allclose(
        original_scores,
        reloaded_scores,
        rtol=1e-7,
        atol=1e-9,
    ):
        raise AssertionError(f"Fresh-load score mismatch: {name}")

quality_gates = {
    "schema_test": "PASS",
    "current_video_id_group_split_test": "PASS",
    "true_video_level_split_test": true_video_status,
    "data_accounting_test": "PASS",
    "exact_duplicate_sha256_split_test": (
        "PASS" if bool(CONFIG["compute_sha256"]) else "DISABLED"
    ),
    "numerical_feature_test": "PASS",
    "classical_ml_smoke_test": "PASS",
    "pytorch_forward_backward_smoke_test": "NOT_APPLICABLE_CLASSICAL_ML",
    "checkpoint_atomic_save_reload_test": "PASS",
    "fresh_load_inference_test": "PASS",
    "test_set_used_for_selection": False,
}

write_json_atomic(
    quality_gates,
    DIRS["metrics"] / "quality_gates.json",
)

print(json.dumps(quality_gates, indent=2))


{
  "schema_test": "PASS",
  "current_video_id_group_split_test": "PASS",
  "true_video_level_split_test": "NOT_VERIFIABLE_FROM_CURRENT_METADATA",
  "data_accounting_test": "PASS",
  "exact_duplicate_sha256_split_test": "PASS",
  "numerical_feature_test": "PASS",
  "classical_ml_smoke_test": "PASS",
  "pytorch_forward_backward_smoke_test": "NOT_APPLICABLE_CLASSICAL_ML",
  "checkpoint_atomic_save_reload_test": "PASS",
  "fresh_load_inference_test": "PASS",
  "test_set_used_for_selection": false
}



## Cell 12 — Final Test

Bu noktaya kadar test seti model/threshold seçimi için kullanılmamıştır. Şimdi üç validation-finalized model aynı sabit test setinde raporlama amacıyla değerlendirilir. Her modelin threshold'u validation aşamasında sabitlenmiştir.


In [13]:

# ============================================================
# CELL 12 — FINAL TEST: THREE PREDECLARED MODEL FAMILIES
# ============================================================

# Transform test only now.
X_test_base_scaled = base_scaler.transform(X_test_base)
X_test_fused_scaled = fused_scaler.transform(X_test_fused)

def evaluate_final_family(name, scores, threshold):
    metrics, pred = metrics_from_scores(y_test, scores, threshold)

    predictions = test_meta[
        [
            "sample_id",
            "label",
            "split",
            "video_id",
            "source_frame",
            "frame_index",
            "face_index",
            "output_path",
        ]
    ].copy()

    predictions["y_true"] = y_test
    predictions["score_fake"] = np.asarray(scores, dtype=float)
    predictions["threshold"] = float(threshold)
    predictions["y_pred"] = pred
    predictions["correct"] = predictions["y_true"] == predictions["y_pred"]
    predictions["model_family"] = name

    return metrics, predictions

baseline_test_scores = baseline_best["model"].decision_function(
    X_test_base_scaled
)
baseline_test_metrics, baseline_pred_df = evaluate_final_family(
    "HOG + LBP -> RBF-SVM",
    baseline_test_scores,
    baseline_best["threshold"],
)

fused_test_scores = fused_best["model"].decision_function(
    X_test_fused_scaled
)
fused_test_metrics, fused_pred_df = evaluate_final_family(
    "HOG + LBP + KAZE -> RBF-SVM",
    fused_test_scores,
    fused_best["threshold"],
)

rf_test_scores = rf_best["model"].predict_proba(X_test_fused)[:, 1]
rf_test_metrics, rf_pred_df = evaluate_final_family(
    "HOG + LBP + KAZE -> Random Forest",
    rf_test_scores,
    rf_best["threshold"],
)

final_metrics_df = pd.DataFrame([
    {
        "model": "HOG + LBP -> RBF-SVM",
        "threshold": baseline_best["threshold"],
        **baseline_test_metrics,
    },
    {
        "model": "HOG + LBP + KAZE -> RBF-SVM",
        "threshold": fused_best["threshold"],
        **fused_test_metrics,
    },
    {
        "model": "HOG + LBP + KAZE -> Random Forest",
        "threshold": rf_best["threshold"],
        **rf_test_metrics,
    },
])

all_predictions_df = pd.concat(
    [baseline_pred_df, fused_pred_df, rf_pred_df],
    ignore_index=True,
)

write_csv_atomic(
    final_metrics_df,
    DIRS["metrics"] / "final_test_metrics.csv",
)
write_csv_atomic(
    all_predictions_df,
    DIRS["predictions"] / "test_predictions_all_models.csv",
)

print("=" * 78)
print("FINAL TEST METRICS")
print("=" * 78)
display(final_metrics_df)


FINAL TEST METRICS


,model,threshold,accuracy,balanced_accuracy,precision,recall,f1,specificity,roc_auc,average_precision,tn,fp,fn,tp
0,HOG + LBP -> RBF-SVM,-0.377563,0.569536,0.562698,0.560748,0.769231,0.648649,0.356164,0.615297,0.647115,52,94,36,120
1,HOG + LBP + KAZE -> RBF-SVM,-0.635737,0.592715,0.582280,0.566802,0.897436,0.694789,0.267123,0.601159,0.574525,39,107,16,140
2,HOG + LBP + KAZE -> Random Forest,0.424118,0.592715,0.583158,0.569038,0.871795,0.688608,0.294521,0.600457,0.588514,43,103,20,136


In [14]:

# ============================================================
# CELL 13 — OPTIONAL GROUP-LEVEL AGGREGATION
# NOT TRUE VIDEO-LEVEL PERFORMANCE
# ============================================================

def group_level_metrics(predictions_df):
    grouped = (
        predictions_df
        .groupby(["video_id", "label"], as_index=False)
        .agg(
            score_fake=("score_fake", "mean"),
            threshold=("threshold", "first"),
        )
    )

    grouped["y_true"] = (grouped["label"] == "fake").astype(int)
    grouped["y_pred"] = (
        grouped["score_fake"] >= grouped["threshold"]
    ).astype(int)

    if grouped["y_true"].nunique() < 2:
        return None, grouped

    metrics, _ = metrics_from_scores(
        grouped["y_true"].to_numpy(),
        grouped["score_fake"].to_numpy(),
        float(grouped["threshold"].iloc[0]),
    )
    return metrics, grouped

group_metric_rows = []
group_prediction_frames = []

for model_name, frame in all_predictions_df.groupby("model_family"):
    metrics, grouped = group_level_metrics(frame)
    grouped["model_family"] = model_name
    group_prediction_frames.append(grouped)

    if metrics is not None:
        group_metric_rows.append(
            {"model": model_name, **metrics}
        )

group_metrics_df = pd.DataFrame(group_metric_rows)
group_predictions_df = pd.concat(
    group_prediction_frames,
    ignore_index=True,
)

write_csv_atomic(
    group_metrics_df,
    DIRS["metrics"] / "metadata_group_level_metrics.csv",
)
write_csv_atomic(
    group_predictions_df,
    DIRS["predictions"] / "metadata_group_level_predictions.csv",
)

print(
    "IMPORTANT: These are metadata video_id GROUP metrics, "
    "not verified original-video metrics."
)
display(group_metrics_df)


IMPORTANT: These are metadata video_id GROUP metrics, not verified original-video metrics.


,model,accuracy,balanced_accuracy,precision,recall,f1,specificity,roc_auc,average_precision,tn,fp,fn,tp
0,HOG + LBP + KAZE -> RBF-SVM,0.5,0.5,0.5,1.0,0.666667,0.0,1.0,1.0,0,1,0,1
1,HOG + LBP + KAZE -> Random Forest,0.5,0.5,0.5,1.0,0.666667,0.0,1.0,1.0,0,1,0,1
2,HOG + LBP -> RBF-SVM,0.5,0.5,0.5,1.0,0.666667,0.0,1.0,1.0,0,1,0,1


In [15]:

# ============================================================
# CELL 14 — REPORT FIGURES (ENGLISH, >= 600 PX SHORT EDGE)
# ============================================================

DPI = int(CONFIG["figure_dpi"])
MIN_SHORT_EDGE = int(CONFIG["min_figure_short_edge_px"])

def save_figure(fig, filename):
    path = DIRS["figures"] / filename
    fig.tight_layout()
    fig.savefig(path, dpi=DPI, bbox_inches="tight")
    plt.close(fig)

    with Image.open(path) as im:
        if min(im.size) < MIN_SHORT_EDGE:
            raise AssertionError(
                f"Figure resolution too small: {path} -> {im.size}"
            )
    return path

# 1) Dataset distribution
dist = (
    eligible.groupby(["split", "label"]).size().unstack(fill_value=0)
    .reindex(["train", "val", "test"])
)
fig, ax = plt.subplots(figsize=(10, 6), dpi=DPI)
x = np.arange(len(dist.index))
width = 0.36
ax.bar(x - width/2, dist.get("real", 0), width, label="REAL")
ax.bar(x + width/2, dist.get("fake", 0), width, label="FAKE")
ax.set_xticks(x, [s.upper() for s in dist.index])
ax.set_title("Eye ROI Dataset Distribution")
ax.set_xlabel("Split")
ax.set_ylabel("Sample Count")
ax.legend()
ax.grid(True, axis="y", alpha=0.25)
save_figure(fig, "01_dataset_distribution.png")

# 2) Validation F1 comparison
plot_search = all_validation_search.copy()
plot_search["label"] = (
    plot_search["family"].astype(str)
    + " | "
    + plot_search["candidate"].astype(int).astype(str)
)
fig, ax = plt.subplots(figsize=(12, 7), dpi=DPI)
ax.bar(plot_search["label"], plot_search["val_f1"])
ax.set_title("Validation F1 by Model Candidate")
ax.set_xlabel("Candidate")
ax.set_ylabel("Validation F1")
ax.tick_params(axis="x", rotation=70)
ax.grid(True, axis="y", alpha=0.25)
save_figure(fig, "02_validation_f1_candidates.png")

# 3) Final metrics
metric_names = [
    "accuracy",
    "precision",
    "recall",
    "specificity",
    "f1",
    "roc_auc",
    "average_precision",
]
fig, ax = plt.subplots(figsize=(12, 7), dpi=DPI)
x = np.arange(len(metric_names))
width = 0.24

for idx, (_, row) in enumerate(final_metrics_df.iterrows()):
    values = [row[m] for m in metric_names]
    ax.bar(
        x + (idx - 1) * width,
        values,
        width,
        label=row["model"],
    )

ax.set_xticks(
    x,
    [
        "Accuracy",
        "Precision",
        "Recall",
        "Specificity",
        "F1",
        "ROC-AUC",
        "Avg. Precision",
    ],
)
ax.set_ylim(0, 1.05)
ax.set_title("Final Test Metrics by Model")
ax.set_ylabel("Score")
ax.tick_params(axis="x", rotation=25)
ax.legend(fontsize=8)
ax.grid(True, axis="y", alpha=0.25)
save_figure(fig, "03_final_test_metrics.png")

# 4–6) Confusion matrices
model_items = [
    ("baseline", baseline_test_metrics),
    ("fused_svm", fused_test_metrics),
    ("random_forest", rf_test_metrics),
]
for index, (short_name, metrics) in enumerate(model_items, start=4):
    cm = np.array(
        [
            [metrics["tn"], metrics["fp"]],
            [metrics["fn"], metrics["tp"]],
        ]
    )
    fig, ax = plt.subplots(figsize=(7, 6), dpi=DPI)
    image = ax.imshow(cm)
    ax.set_title(f"Confusion Matrix — {short_name.replace('_', ' ').title()}")
    ax.set_xlabel("Predicted Class")
    ax.set_ylabel("True Class")
    ax.set_xticks([0, 1], ["REAL", "FAKE"])
    ax.set_yticks([0, 1], ["REAL", "FAKE"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=12)
    fig.colorbar(image, ax=ax)
    save_figure(fig, f"{index:02d}_confusion_matrix_{short_name}.png")

# 7) ROC curves
fig, ax = plt.subplots(figsize=(10, 7), dpi=DPI)
for model_name, scores in [
    ("HOG + LBP -> RBF-SVM", baseline_test_scores),
    ("HOG + LBP + KAZE -> RBF-SVM", fused_test_scores),
    ("HOG + LBP + KAZE -> Random Forest", rf_test_scores),
]:
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc_value = roc_auc_score(y_test, scores)
    ax.plot(fpr, tpr, linewidth=2, label=f"{model_name} (AUC={auc_value:.3f})")

ax.plot([0, 1], [0, 1], linestyle="--", label="Random")
ax.set_title("ROC Curves — Final Test")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.25)
save_figure(fig, "07_roc_curves.png")

# 8) Precision–Recall
fig, ax = plt.subplots(figsize=(10, 7), dpi=DPI)
for model_name, scores in [
    ("HOG + LBP -> RBF-SVM", baseline_test_scores),
    ("HOG + LBP + KAZE -> RBF-SVM", fused_test_scores),
    ("HOG + LBP + KAZE -> Random Forest", rf_test_scores),
]:
    precision, recall, _ = precision_recall_curve(y_test, scores)
    ap = average_precision_score(y_test, scores)
    ax.plot(recall, precision, linewidth=2, label=f"{model_name} (AP={ap:.3f})")

ax.set_title("Precision–Recall Curves — Final Test")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.25)
save_figure(fig, "08_precision_recall_curves.png")

# 9) Score distributions
fig, ax = plt.subplots(figsize=(11, 7), dpi=DPI)
for model_name, scores in [
    ("Baseline SVM", baseline_test_scores),
    ("KAZE SVM", fused_test_scores),
    ("Random Forest", rf_test_scores),
]:
    ax.hist(scores[y_test == 0], bins=30, alpha=0.25, label=f"{model_name} REAL")
    ax.hist(scores[y_test == 1], bins=30, alpha=0.25, label=f"{model_name} FAKE")

ax.set_title("Final Test Score Distributions")
ax.set_xlabel("Model Score")
ax.set_ylabel("Frequency")
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.25)
save_figure(fig, "09_score_distributions.png")

figure_audit = []
for p in sorted(DIRS["figures"].glob("*.png")):
    with Image.open(p) as im:
        figure_audit.append({
            "file": p.name,
            "width_px": im.size[0],
            "height_px": im.size[1],
            "quality_pass": min(im.size) >= MIN_SHORT_EDGE,
        })

figure_audit_df = pd.DataFrame(figure_audit)
write_csv_atomic(
    figure_audit_df,
    DIRS["metrics"] / "figure_quality_audit.csv",
)

print(figure_audit_df)


                                    file  width_px  height_px  quality_pass
0            01_dataset_distribution.png      1485        885          True
1        02_validation_f1_candidates.png      1785       1035          True
2              03_final_test_metrics.png      1785       1035          True
3       04_confusion_matrix_baseline.png       993        865          True
4      05_confusion_matrix_fused_svm.png       993        865          True
5  06_confusion_matrix_random_forest.png       993        857          True
6                      07_roc_curves.png      1485       1035          True
7         08_precision_recall_curves.png      1485       1035          True
8             09_score_distributions.png      1632       1035          True


In [16]:

# ============================================================
# CELL 15 — RUN SUMMARY + OUTPUT MANIFEST + FINAL QUALITY GATE
# ============================================================

selected_validation = {
    "hog_lbp_rbf_svm": {
        "C": baseline_best["C"],
        "threshold": baseline_best["threshold"],
        "validation_metrics": baseline_best["validation_metrics"],
    },
    "hog_lbp_kaze_rbf_svm": {
        "C": fused_best["C"],
        "threshold": fused_best["threshold"],
        "validation_metrics": fused_best["validation_metrics"],
    },
    "hog_lbp_kaze_random_forest": {
        "configuration": rf_best["configuration"],
        "threshold": rf_best["threshold"],
        "validation_metrics": rf_best["validation_metrics"],
    },
}

run_summary = {
    "run_id": RUN_ID,
    "status": "COMPLETED",
    "experiment": "Eye ROI HOG + LBP + KAZE -> SVM / Random Forest",
    "region": "eye",
    "roi": "combined_eye",
    "positive_class": "fake",
    "seed": SEED,
    "paths": {
        "input_data": str(DATA_ROOT),
        "roi_metadata": str(ROI_METADATA_PATH),
        "output_run": str(RUN_DIR),
    },
    "data": accounting,
    "feature_dimensions": feature_dimensions,
    "validation_selection": selected_validation,
    "test_metrics": final_metrics_df.to_dict(orient="records"),
    "metadata_group_level_metrics": group_metrics_df.to_dict(orient="records"),
    "quality_gates": quality_gates,
    "figure_count": int(len(list(DIRS["figures"].glob("*.png")))),
    "checkpoint_note": (
        "Classical ML experiment: SVC/RF models, train-fitted scalers, "
        "validation-selected thresholds, config and RNG state are atomically "
        "serialized and fresh-load tested. PyTorch optimizer/epoch state is "
        "not applicable."
    ),
    "completed_at": datetime.now(
        ZoneInfo("Europe/Istanbul")
    ).isoformat(),
}

write_json_atomic(
    run_summary,
    RUN_DIR / "run_summary.json",
)

# Output manifest with SHA256 for all final files except manifest itself.
manifest_rows = []
for path in sorted(RUN_DIR.rglob("*")):
    if not path.is_file():
        continue
    if path.name == "output_manifest.csv":
        continue

    manifest_rows.append({
        "relative_path": path.relative_to(RUN_DIR).as_posix(),
        "size_bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    })

output_manifest = pd.DataFrame(manifest_rows)
write_csv_atomic(
    output_manifest,
    RUN_DIR / "output_manifest.csv",
)

required_outputs = [
    RUN_DIR / "config_resolved.yaml",
    RUN_DIR / "requirements_lock.txt",
    RUN_DIR / "environment.json",
    RUN_DIR / "run_summary.json",
    RUN_DIR / "output_manifest.csv",
    DIRS["metrics"] / "data_accounting.json",
    DIRS["metrics"] / "validation_model_search.csv",
    DIRS["metrics"] / "final_test_metrics.csv",
    DIRS["metrics"] / "quality_gates.json",
    DIRS["predictions"] / "test_predictions_all_models.csv",
    DIRS["checkpoints"] / "hog_lbp_rbf_svm.joblib",
    DIRS["checkpoints"] / "hog_lbp_kaze_rbf_svm.joblib",
    DIRS["checkpoints"] / "hog_lbp_kaze_random_forest.joblib",
]

missing_final = [
    str(p) for p in required_outputs
    if not p.is_file() or p.stat().st_size == 0
]
if missing_final:
    raise RuntimeError(
        "Final output integrity failure:\n- "
        + "\n- ".join(missing_final)
    )

if not figure_audit_df["quality_pass"].all():
    raise AssertionError("One or more figures failed resolution quality gate.")

print("=" * 78)
print("EXPERIMENT COMPLETED SUCCESSFULLY")
print("=" * 78)
print("Run ID :", RUN_ID)
print("Output :", RUN_DIR)
print("Figures:", run_summary["figure_count"])
print("\nFinal test summary:")
display(final_metrics_df)


EXPERIMENT COMPLETED SUCCESSFULLY
Run ID : 20260807_1719_eye_hog_lbp_kaze_svm_rf_seed42
Output : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Kader/Deney 1/Sonuçlar/20260807_1719_eye_hog_lbp_kaze_svm_rf_seed42
Figures: 9

Final test summary:


,model,threshold,accuracy,balanced_accuracy,precision,recall,f1,specificity,roc_auc,average_precision,tn,fp,fn,tp
0,HOG + LBP -> RBF-SVM,-0.377563,0.569536,0.562698,0.560748,0.769231,0.648649,0.356164,0.615297,0.647115,52,94,36,120
1,HOG + LBP + KAZE -> RBF-SVM,-0.635737,0.592715,0.582280,0.566802,0.897436,0.694789,0.267123,0.601159,0.574525,39,107,16,140
2,HOG + LBP + KAZE -> Random Forest,0.424118,0.592715,0.583158,0.569038,0.871795,0.688608,0.294521,0.600457,0.588514,43,103,20,136



## Notebook Çıktı Yapısı

```text
Kader/
└── Deney 1/
    └── Sonuçlar/
        └── YYYYMMDD_HHMM_eye_hog_lbp_kaze_svm_rf_seed42/
            ├── checkpoints/
            │   ├── hog_lbp_rbf_svm.joblib
            │   ├── hog_lbp_kaze_rbf_svm.joblib
            │   └── hog_lbp_kaze_random_forest.joblib
            ├── logs/
            │   └── run.log
            ├── metrics/
            │   ├── data_accounting.json
            │   ├── validation_model_search.csv
            │   ├── validation_selected_families.csv
            │   ├── final_test_metrics.csv
            │   ├── metadata_group_level_metrics.csv
            │   ├── quality_gates.json
            │   └── figure_quality_audit.csv
            ├── predictions/
            │   ├── test_predictions_all_models.csv
            │   └── metadata_group_level_predictions.csv
            ├── figures/
            │   └── 9 adet rapor grafiği
            ├── artifacts/
            │   ├── metadata_used.csv
            │   ├── features_train.npz
            │   ├── features_val.npz
            │   ├── features_test.npz
            │   └── feature_dimensions.json
            ├── config_resolved.yaml
            ├── requirements_lock.txt
            ├── environment.json
            ├── output_manifest.csv
            └── run_summary.json
```

### Bilimsel not
Mevcut Eye ROI metadata'sındaki `video_id` orijinal kaynak videoyu kesin olarak kanıtlamıyorsa, notebook group-level aggregation üretebilir ancak bunu **gerçek video-level performans** olarak adlandırmaz.
